In [ ]:
%pip install -q ipywidgets
# scipy ships with the default JupyterLite (Pyodide) kernel, no install needed

# Particle stability in yield stress fluids

Yield stress can trap a particle of a given size and density: if the stress the particle
exerts on the surrounding structure stays below the fluid's yield stress, the particle
does not settle. The suspension criterion from Beris, Tsamopoulos, Armstrong & Brown (1985) is:

$$ \sigma_{y,min}=\frac{F}{14 \pi r^2} $$

where $F$ is the net gravitational force on a single particle:

$$F = \frac{4}{3} \pi r^3 g \, (\rho_p - \rho_l)$$

which gives the minimum yield stress required to suspend the particle:

$$ \sigma_{y,min}=\frac{2\, r\, g\, (\rho_p - \rho_l)}{21} $$

or, inverted, the largest particle a given yield stress can trap:

$$r_{max}=\frac{21\, \sigma_y}{2\, g\, (\rho_p - \rho_l)}$$

In [ ]:
# Minimum yield stress to suspend a particle / largest trappable particle
from scipy import constants

def net_force(radius, drho):
    """Net gravitational force on a spherical particle.
    radius: particle radius [m]; drho: |rho_particle - rho_fluid| [kg/m^3] -> force [N]"""
    return drho * constants.g * 4.0/3.0 * constants.pi * radius**3

def min_yield_stress(radius, drho):
    """Minimum yield stress [Pa] to trap a particle of `radius` [m]."""
    return net_force(radius, drho) / (14 * constants.pi * radius**2)

def max_radius(ystress, drho):
    """Largest particle radius [m] trappable by a fluid of yield stress `ystress` [Pa]."""
    return ystress * 21 / (2 * constants.g * drho)

def vel_term(radius, drho=1000.0, visc=0.1):
    """Stokes terminal velocity [m/s] of the particle in a Newtonian fluid of viscosity `visc` [Pa s]."""
    return (2.0/9.0) * drho / visc * constants.g * radius**2


In [ ]:
# Sanity check (also runs headless): 50 um particle, drho = 1000 kg/m^3
r, drho = 50e-6, 1000.0
print(f"r = {r*1e6:.0f} um, drho = {drho:.0f} kg/m^3")
print(f"  min yield stress : {min_yield_stress(r, drho):.4g} Pa")
print(f"  r_max at 0.1 Pa  : {max_radius(0.1, drho)*1e6:.1f} um")
print(f"  Stokes velocity  : {vel_term(r, drho)*1e3:.3g} mm/s (in 0.1 Pa s oil)")


## Interactive calculator

Enter a particle radius and density mismatch, press the button:

In [ ]:
import ipywidgets as widgets
from IPython.display import display

style = {'description_width': 'initial'}
radius_w = widgets.FloatText(value=50e-6, description='Particle radius [m]:', style=style)
drho_w = widgets.FloatText(value=1000.0, description='Density mismatch [kg/m^3]:', style=style)
run_w = widgets.Button(description='Compute required yield stress')
out_w = widgets.Output()

def _on_run(_):
    with out_w:
        out_w.clear_output()
        try:
            s = min_yield_stress(float(radius_w.value), float(drho_w.value))
            print(f"Minimum yield stress required: {s:.4g} Pa")
        except Exception as e:
            print(f"Could not compute: {e}")

run_w.on_click(_on_run)
display(widgets.VBox([radius_w, drho_w, run_w, out_w]))


## Inverse calculation

Largest particle that a fluid with a known yield stress can keep suspended:

In [ ]:
import ipywidgets as widgets
from IPython.display import display

style = {'description_width': 'initial'}
sy_w = widgets.FloatText(value=0.1, description='Yield stress [Pa]:', style=style)
drho2_w = widgets.FloatText(value=1000.0, description='Density mismatch [kg/m^3]:', style=style)
run2_w = widgets.Button(description='Compute max trappable radius')
out2_w = widgets.Output()

def _on_run2(_):
    with out2_w:
        out2_w.clear_output()
        try:
            rmax = max_radius(float(sy_w.value), float(drho2_w.value))
            print(f"Maximum trappable radius: {rmax:.4g} m ({rmax*1e6:.2g} um)")
        except Exception as e:
            print(f"Could not compute: {e}")

run2_w.on_click(_on_run2)
display(widgets.VBox([sy_w, drho2_w, run2_w, out2_w]))


# References

- Beris, A. N., Tsamopoulos, J. A., Armstrong, R. C. & Brown, R. A. (1985). _Creeping motion of a sphere through a Bingham plastic._ Journal of Fluid Mechanics **158**, 219–244. [doi:10.1017/S0022112085002622](https://doi.org/10.1017/S0022112085002622)
